In [0]:
%sql
-- SCRIPT: 6.1_analise_absenteismo
-- LOCAL: 3_gold/src/6_monitor_presenca/
-- OBJETIVO: Monitora as ausencias. Usa as tabelas gold_atlas_frentes_parlamentares e bronze_presenca_votacoes (gerada pela 1_bronze..6.1_ingestao_presencas_votacoes)
-- ENTREGA: Desenvolver uma visão sobre junção entre eventos e votações para medir ausências em votações

CREATE OR REPLACE TABLE gold_monitor_absenteismo AS
WITH total_deputados AS (
    SELECT DISTINCT id_deputado, nome_deputado, partido, uf
    FROM gold_atlas_frentes_parlamentares
),
lista_votacoes AS (
    SELECT DISTINCT id_votacao FROM bronze_presenca_votacoes
),
matriz_presenca AS (
    SELECT * FROM total_deputados CROSS JOIN lista_votacoes
)

SELECT 
    m.id_votacao,
    m.id_deputado,
    m.nome_deputado,
    m.partido,
    m.uf,    
    COALESCE(b.data_captura, '2026-05-10') as data_voto, 
    CASE 
        WHEN b.id_deputado IS NULL THEN 'AUSENTE' 
        ELSE 'PRESENTE' 
    END AS status_presenca
FROM matriz_presenca m
LEFT JOIN bronze_presenca_votacoes b 
  ON m.id_deputado = b.id_deputado 
  AND m.id_votacao = b.id_votacao;

SELECT * FROM gold_monitor_absenteismo LIMIT 10;